# 08 — Low Voltage Current Mirror (FVF-based)

**Dataset:** `datasets/low_voltage_cmirror/`

Topology: 8 NMOS — 2 FVF stages for bias generation + 4 output transistors
- FVF_cascode: sets cascode gate bias (VF1)
- FVF_bias:    sets output gate bias (IBIAS2)
- 2× output bottom + 2× output cascode

Nodes: `IBIAS1 IBIAS2 IOUT1 IOUT2 GND`

In [ ]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel


In [ ]:
ibias1 = gl.Net('ibias1')   # high bias input
ibias2 = gl.Net('ibias2')   # low bias output of FVF_bias
iout1  = gl.Net('iout1')
iout2  = gl.Net('iout2')
vb_cas = gl.Net('vb_cas')   # FVF_cascode feedback node
vb_bia = gl.Net('vb_bia')   # FVF_bias feedback node
vf1    = gl.Net('vf1')      # FVF_cascode output (drives output-bottom gates)
vmid1  = gl.Net('vmid1')    # output1 mid-node
vmid2  = gl.Net('vmid2')    # output2 mid-node

# FVF_cascode — provides gate bias for output bottom transistors
m1 = gl.nmos(w=4.15, g=ibias1, d=vb_cas, s=vf1)     # G=IBIAS1, D=Ib, S=VF1
m2 = gl.nmos(w=4.15, g=vb_cas, d=vf1,    s=gl.gnd)  # feedback

# FVF_bias — provides cascode gate bias (IBIAS2 node)
m3 = gl.nmos(w=4.15, g=ibias1, d=vb_bia, s=ibias2)   # G=IBIAS1, D=Ib, S=IBIAS2
m4 = gl.nmos(w=1.42, g=vb_bia, d=ibias2, s=gl.gnd)  # narrow W → large Vgs offset

# Output bottom transistors (gate driven by FVF_cascode output VF1)
m5 = gl.nmos(w=4.15, g=vf1, d=vmid1, s=gl.gnd)
m6 = gl.nmos(w=4.15, g=vf1, d=vmid2, s=gl.gnd)

# Output cascode transistors (gate driven by FVF_bias output IBIAS2)
m7 = gl.nmos(w=4.15, g=ibias2, d=iout1, s=vmid1)
m8 = gl.nmos(w=4.15, g=ibias2, d=iout2, s=vmid2)

chip = gl.build(m1, m2, m3, m4, m5, m6, m7, m8, name='low_voltage_cmirror')
chip.show()
chip.drc()
chip.sim()